[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cuny-sps-datascience/citibike-data-pipeline/blob/main/teams/team-6/mvp/play_model1.ipynb)

# Play with Model 1 (calendar-only LightGBM)

Run cells top to bottom (Shift+Enter). Change the values in the **CHANGE ME** lines and re-run that cell.

- **1** load everything
- **2** pick a station + week → predicted vs actual
- **3** where does it miss? (error by hour of day)
- **4** worst days — the misses weather should fix
- **5** ask the model about any date/hour
- **6** experiment: turn features off, retrain, compare — scored on **validation** (Jan–Feb 2026); the test period is only for reporting the final model (`scripts/train_mvp.py`)

**Run it:** click the Colab badge (nothing to install), or locally: `pip install -r requirements.txt` then `jupyter lab` from this folder.

*Before PR #72 is merged, use this Colab link instead:* [open from the PR branch](https://colab.research.google.com/github/jamilton08/citibike-data-pipeline/blob/jonathan/mvp-6-stations-model1/teams/team-6/mvp/play_model1.ipynb)

In [ ]:
# 0. Setup — only does something in Colab / when the data isn't next to the notebook
import os, shutil, subprocess, sys
if not os.path.exists('model1/predictions.parquet'):
    os.chdir('/content')
    target = '/content/cbp'
    sources = [('https://github.com/cuny-sps-datascience/citibike-data-pipeline.git', 'main'),
               ('https://github.com/jamilton08/citibike-data-pipeline.git', 'jonathan/mvp-6-stations-model1')]
    for repo, branch in sources:
        shutil.rmtree(target, ignore_errors=True)          # clear leftovers from earlier runs
        r = subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', branch, repo, target], capture_output=True, text=True)
        if r.returncode == 0 and os.path.exists(f'{target}/teams/team-6/mvp/model1/predictions.parquet'):
            print(f'using {branch} from {repo}')
            break
        print(f'MVP files not on {branch} yet, trying next source')
    else:
        raise RuntimeError('Could not find the MVP files on main or on the PR branch')
    os.chdir(f'{target}/teams/team-6/mvp')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm', 'pyarrow'], check=True)
print('working dir:', os.getcwd())

In [ ]:
# 1. Load data, model, predictions — same rows/features as scripts/train_mvp.py (via scripts/mvp_data.py)
import warnings; warnings.filterwarnings("ignore")
import sys, pandas as pd, numpy as np, lightgbm as lgb, matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
sys.path.insert(0, 'scripts')
from mvp_data import ZONE, HOLIDAYS, FEATURES, TARGET, load_hourly, build_frame
plt.rcParams['figure.figsize'] = (12, 4)

pred = pd.read_parquet('model1/predictions.parquet')
pred['target_local'] = pd.to_datetime(pred.target_hour_utc, utc=True).dt.tz_convert(ZONE)
booster = lgb.Booster(model_file='model1/lgbm_calendar.txt')
hourly = load_hourly()

STATIONS = dict(pred.drop_duplicates('station_id')[['station_name','station_id']].astype(str).values)
print('stations:'); [print('  ', n) for n in STATIONS]
print('predictions:', len(pred), 'rows |', pred.split.value_counts().to_dict())

In [ ]:
# 2. One station, one week: actual vs LightGBM vs baseline
STATION = 'W 21 St & 6 Ave'      # CHANGE ME (any name printed above)
WEEK_START = '2026-06-08'        # CHANGE ME (a Monday between 2026-01-05 and 2026-08-24)

w = pred[(pred.station_name == STATION) & (pred.horizon == 1)]
w = w[(w.target_local >= pd.Timestamp(WEEK_START, tz=ZONE)) & (w.target_local < pd.Timestamp(WEEK_START, tz=ZONE) + pd.Timedelta(days=7))]
plt.bar(w.target_local, w.departures, width=1/24, color='lightgray', label='actual')
plt.plot(w.target_local, w.pred_lightgbm, lw=2, label='LightGBM')
plt.plot(w.target_local, w.pred_baseline, lw=1.5, ls='--', label='baseline (avg)')
plt.title(f'{STATION} — week of {WEEK_START}'); plt.ylabel('departures / hour'); plt.legend(); plt.show()
print(f'MAE this week — LightGBM {mean_absolute_error(w.departures, w.pred_lightgbm):.2f} | baseline {mean_absolute_error(w.departures, w.pred_baseline):.2f}')

In [ ]:
# 3. Where does it miss? Average error by hour of day (test period, all stations)
t = pred[(pred.split == 'test') & (pred.horizon == 1)].copy()
t['hour'] = t.target_local.dt.hour
t['err_lgbm'] = (t.pred_lightgbm - t.departures).abs()
t['err_base'] = (t.pred_baseline - t.departures).abs()
t.groupby('hour')[['err_lgbm','err_base']].mean().plot(kind='bar', width=0.8)
plt.title('Average miss by hour of day (test, Mar–Aug 2026)'); plt.ylabel('MAE (bikes)'); plt.xlabel('hour'); plt.show()
print('Rush hours = biggest misses. That is where weather and subway have the most room to help.')

In [ ]:
# 4. Worst days — biggest daily misses in the test period
d = t.assign(date=t.target_local.dt.date).groupby(['date','station_name']).agg(
        actual=('departures','sum'), predicted=('pred_lightgbm','sum')).reset_index()
d['miss'] = d.predicted - d.actual
d['miss_pct'] = (d.miss / d.predicted * 100).round(0)
print('Days the model OVER-predicted most (likely rain / cold / events):')
display(d.sort_values('miss', ascending=False).head(10).round(1))
print('Days it UNDER-predicted most (unexpectedly busy):')
display(d.sort_values('miss').head(10).round(1))
print('Look up a few of these dates — if the over-predicted ones were rainy, that is the case for model 2.')

In [ ]:
# 5. Ask the model: departures for any station / date / hour
STATION = 'Broadway & E 14 St'   # CHANGE ME
WHEN = '2026-07-15 17:00'        # CHANGE ME (local NYC time; any date works, even future)

def features_for(station_name, when_local, horizon=1):
    ts = pd.Timestamp(when_local, tz=ZONE) + pd.Timedelta(hours=horizon - 1)
    row = pd.DataFrame([{'station_id': STATIONS[station_name], 'horizon': horizon, 't_hour': ts.hour, 't_dow': ts.dayofweek,
                         't_month': ts.month, 't_doy': ts.dayofyear, 't_weekend': int(ts.dayofweek >= 5),
                         't_holiday': int(ts.date() in HOLIDAYS)}])
    row['station_id'] = pd.Categorical(row.station_id, categories=sorted(STATIONS.values()))
    return row[FEATURES], ts

for h in (1, 2, 3):
    X, ts = features_for(STATION, WHEN, h)
    print(f'{STATION} — {ts:%a %b %d %Y, %I %p}: ~{booster.predict(X)[0]:.1f} departures')
print('\nTry: a winter morning vs a summer evening, a weekend, a holiday like 2026-07-03.')

In [ ]:
# 6. Experiment: drop features, retrain, compare VALIDATION MAE (~10 s per run)
#    Validation (Jan–Feb 2026) is for choices like this. The test period stays untouched here —
#    its number for the final model comes from scripts/train_mvp.py.
full = build_frame(hourly)
tr, va = full[full.split == 'train'], full[full.split == 'validation']

def run(features):
    m = lgb.LGBMRegressor(objective='poisson', n_estimators=3000, learning_rate=0.03, num_leaves=63, min_child_samples=50,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.9, random_state=622, verbose=-1)
    m.fit(tr[features], tr[TARGET], eval_set=[(va[features], va[TARGET])], callbacks=[lgb.early_stopping(150, verbose=False)])
    return mean_absolute_error(va[TARGET], m.predict(va[features]))

base = tr.groupby(['station_id','t_hour','t_dow'], observed=True)[TARGET].mean().rename('b').reset_index()
base_mae = mean_absolute_error(va[TARGET], va.merge(base, how='left', on=['station_id','t_hour','t_dow']).b)

# CHANGE ME: add/remove experiments
experiments = {
    'all features':            FEATURES,
    'no day-of-year / month':  [f for f in FEATURES if f not in ('t_doy', 't_month')],
    'no holiday':              [f for f in FEATURES if f != 't_holiday'],
    'hour + weekday only':     ['station_id', 't_hour', 't_dow'],
}
for name, feats in experiments.items():
    print(f'{name:28s} validation MAE {run(feats):.3f}')
print(f'\nBaseline (station x hour x weekday avg), validation: {base_mae:.3f}')